# Quickstart to demonstrate a search agent with DeepAgent

In [1]:
!pip install deepagents tavily-python langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 173.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.1/550.1 kB 215.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.5/832.5 kB 215.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 140.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 168.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 192.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 196.5 MB/s  0:00:00
  Attempting uninstall: websockets━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/32 [zstandard]
    Found existing installation: websockets 16.0━━━━━━━━━━━━━━  1/32 [zstandard]
    Uninstalling websockets-16.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/32 [zstandard]
      Successfully uninstalled websockets-16.0━━━━━━━━━━━━━━━━  1/32 [zstandard]
  Attempting uninstall: google-auth╸━━━━━━━━━━━━━━━━━ 18/32 [langsmith]]
    Found existing installation: google-auth 2.47.0━━━━━━━

## Create a search tool

In [2]:
import os
from typing import Literal
from tavily import TavilyClient


In [3]:
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

In [4]:
def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

# Create the Agent

In [5]:
from langchain_openai import ChatOpenAI

qwen = ChatOpenAI(
    base_url=os.environ["QWEN_ENDPOINT"],
    api_key=os.environ["QWEN_API_KEY"],
    model="Qwen3.6-35B-A3B",
)

In [6]:
research_instruction = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

In [7]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=qwen,
    tools=[internet_search],
    system_prompt=research_instruction,
)

# Test the agent

In [8]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]})

# Print the agent's response
print(result["messages"][-1].content)



**LangGraph** is an open-source framework from LangChain for building and managing AI agent workflows using graph-based structures. It models AI workflows as a directed graph composed of **nodes** (steps/functions) and **edges** (connections/transitions between steps).

### Key Concepts

- **Nodes** — Individual steps in the workflow (e.g., calling an LLM, running a tool, making a decision).
- **Edges** — Transitions that define the flow between nodes. Conditional edges enable branching logic based on state.
- **State** — A shared, persistent data structure that flows through the graph and can be updated at each node.
- **Cycles** — Unlike a linear chain, graphs can loop, allowing agents to revisit steps, retry, or iterate.

### Why LangGraph vs. LangChain Chains

LangChain's traditional `Chain` approach follows a **linear, fixed sequence** of steps (input → output, one direction). LangGraph provides:

- **Branching & conditional logic** — The workflow can take different paths based 